In [15]:
import pandas as pd
import numpy as np

In [49]:
df = pd.read_csv('/content/seo_analysis.csv', sep=',', encoding='utf-8', na_values=['', ' '])
def normalize_cell(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null", "n/a", "na", "-", "—"}:
        return np.nan
    return s
clean = df.apply(lambda col: col.map(normalize_cell))
filled_count = clean.notna().sum(axis=1)
rows_to_keep = ~(
    (filled_count == 0) |
    ((filled_count == 1) & clean["request_key"].notna())
)
df = df.loc[rows_to_keep].copy()

df.head()

,request_key,категория,запрос,тип_запроса,домены_источников,тип_источника,есть_ли_pg,конкуренты,кто_выигрывает,пробелы
0,трусики памперс джуниор,baby care,трусики памперс джуниор,брендовый,"aptekiplus.ru, eapteka.ru, stolichki.ru, ozon....",перекуп,Да,NaN,P&G,Недостаточно авторитетных внешних источников
1,pampers premium care,baby care,pampers premium care,брендовый,"goldapple.ru, Detmir.ru, votonia.ru, OZON.ru, ...",маркетплейс,Да,NaN,P&G,Недостаточно авторитетных внешних источников
2,как убрать налет на зубах отзывы,oral care,как убрать налет на зубах отзывы,консультационный,"sprosivracha.com, www.KrasotaiMedicina.ru, alp...",экспертный ресурс,Нет,NaN,конкуренты,"Нет присутствия P&G в AI-ответе, Присутствует ..."
3,гель для душа old spice wolfthorn,hair care,гель для душа old spice wolfthorn,брендовый,"poryadok.ru, lenta.com, goldapple.ru, megaptek...",маркетплейс,Да,NaN,P&G,Недостаточно авторитетных внешних источников
4,как убрать зеленый налет на зубах,oral care,как убрать зеленый налет на зубах,проблемно-решающий,"probolezny.ru, fdc-vip.ru, доктормамонтов.рф, ...",экспертный ресурс,Нет,NaN,одинаково,"Нет присутствия P&G в AI-ответе, Присутствует ..."


In [50]:
domains = df[["запрос", "категория", "тип_запроса", "домены_источников"]].copy()
domains["домены_источников"] = domains["домены_источников"].fillna("").str.split(",")
domains = domains.explode("домены_источников")
domains["домены_источников"] = domains["домены_источников"].str.strip().str.lower()
domains = domains[domains["домены_источников"] != ""]
domains

,запрос,категория,тип_запроса,домены_источников
0,трусики памперс джуниор,baby care,брендовый,aptekiplus.ru
0,трусики памперс джуниор,baby care,брендовый,eapteka.ru
0,трусики памперс джуниор,baby care,брендовый,stolichki.ru
0,трусики памперс джуниор,baby care,брендовый,ozon.ru
0,трусики памперс джуниор,baby care,брендовый,market.yandex.ru
...,...,...,...,...
687,скачать head and shoulders,hair care,брендовый,nsportal.ru
687,скачать head and shoulders,hair care,брендовый,pinkamuz.pro
687,скачать head and shoulders,hair care,брендовый,mp3ferz.cc
687,скачать head and shoulders,hair care,брендовый,muzmind.org


In [51]:
competitors = df[["запрос", "категория", "тип_запроса", "конкуренты"]].copy()
competitors["конкуренты"] = competitors["конкуренты"].fillna("").str.split(",")
competitors = competitors.explode("конкуренты")
competitors["конкуренты"] = competitors["конкуренты"].str.strip().str.lower()
competitors = competitors[competitors["конкуренты"] != ""]
competitors

,запрос,категория,тип_запроса,конкуренты
5,крем под подгузник отзывы,baby care,категорийный,ла-кри
5,крем под подгузник отзывы,baby care,категорийный,lulu
5,крем под подгузник отзывы,baby care,категорийный,моё солнышко
7,шампунь от перхоти клеар,hair care,брендовый,clear
14,NaN,baby care,категорийный,joonies premium soft
...,...,...,...,...
674,NaN,oral care,категорийный,innova
685,шампунь для волос 2 в 1,hair care,категорийный,dove
685,шампунь для волос 2 в 1,hair care,категорийный,ollin professional
685,шампунь для волос 2 в 1,hair care,категорийный,fructis (garnier)


In [52]:
pg_stats = (
    df["есть_ли_pg"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"nan": None, "": None})
)

pg_stats = (
    pg_stats[pg_stats.isin(["да", "нет"])]
    .value_counts()
    .rename_axis("есть_ли_pg")
    .reset_index(name="count")
)

pg_stats["proportion"] = pg_stats["count"] / pg_stats["count"].sum()
pg_stats["proportion_pct"] = (pg_stats["proportion"] * 100).round(1).astype(str) + "%"

pg_stats


,есть_ли_pg,count,proportion,proportion_pct
0,нет,223,0.585302,58.5%
1,да,158,0.414698,41.5%


In [53]:
df["кто_выигрывает"].value_counts(normalize=True) * 100

,proportion
кто_выигрывает,
конкуренты,48.695652
P&G,37.391304
одинаково,13.913043


In [57]:
min_pct = 1

domain_share = (
    domains.groupby("домены_источников")["запрос"]
    .nunique()
    .div(df["запрос"].nunique())
    .mul(100)
    .sort_values(ascending=False)
)

domain_share = domain_share[domain_share >= min_pct]

domain_share.map(lambda x: f"{x:.2f}%")
# если процент запросов в которых встречается домен слииишком мал (меньше 1) - не учитываб

,запрос
домены_источников,
market.yandex.ru,35.61%
ozon.ru,23.44%
поиск по всему интернету,20.77%
apteka.ru,16.32%
wildberries.ru,13.06%
...,...
markakachestva.ru,1.19%
linline-clinic.ru,1.19%
vichy.ru,1.19%


In [58]:
df["тип_источника"].value_counts(normalize=True) * 100

,proportion
тип_источника,
маркетплейс,41.994751
экспертный ресурс,22.834646
обзор,22.572178
сайт бренда,4.199475
форум,2.624672
перекуп,2.099738
"перекуп, маркетплейс",0.524934
"маркетплейс, форум",0.524934
"форум, маркетплейс",0.524934


In [59]:
pd.crosstab(df["тип_запроса"], df["есть_ли_pg"], normalize="index") * 100

есть_ли_pg,Да,Нет
тип_запроса,,
брендовый,90.225564,9.774436
категорийный,21.804511,78.195489
консультационный,6.451613,93.548387
проблемно-решающий,13.636364,86.363636


In [60]:
df["пробелы_clean"] = (
    df["пробелы"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
df["пробелы_clean"]

,пробелы_clean
0,недостаточно авторитетных внешних источников
1,недостаточно авторитетных внешних источников
2,"нет присутствия p&g в ai-ответе, присутствует ..."
3,недостаточно авторитетных внешних источников
4,"нет присутствия p&g в ai-ответе, присутствует ..."
...,...
680,"нет присутствия p&g в ai-ответе, присутствует ..."
681,недостаточно авторитетных внешних источников
682,"нет объяснения критериев выбора, недостаточно ..."
685,"нет присутствия p&g в ai-ответе, присутствует ..."


In [66]:
df["пробелы_clean"] = (
    df["пробелы"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
GAP_PATTERNS = {
    "no_pg_in_ai": r"нет присутствия p&g в ai-ответе",
    "pg_not_leading": r"p&g есть,\s*но не в лидирующей рекомендации",
    "no_category_content": r"присутствует только в брендовых запросах,\s*но отсутствует в категорийных",
    "no_consultation_content": r"присутствует в категорийных,\s*но отсутствует в консультационных",
    "no_comparison_content": r"нет контента под сравнение с конкурентами",
    "no_selection_criteria": r"нет объяснения критериев выбора",
    "weak_authority_sources": r"недостаточно авторитетных внешних источников",
    "weak_purchase_platform_presence": r"слабое присутствие на площадках,\s*влияющих на покупку",
    "competitor_problem_solution_better": r"у конкурента лучше покрыт problem-solution сценарий",
    "competitor_comparison_better": r"у конкурента лучше покрыт comparison-сценарий",
}
for gap_name, pattern in GAP_PATTERNS.items():
    df[gap_name] = df["пробелы_clean"].str.contains(pattern, regex=True, na=False).astype(int)

gap_cols = list(GAP_PATTERNS.keys())

df["has_any_gap"] = (df[gap_cols].sum(axis=1) > 0).astype(int)
df["gap_count"] = df[gap_cols].sum(axis=1)

df.head()



,request_key,категория,запрос,тип_запроса,домены_источников,тип_источника,есть_ли_pg,конкуренты,кто_выигрывает,пробелы,...,no_category_content,no_consultation_content,no_comparison_content,no_selection_criteria,weak_authority_sources,weak_purchase_platform_presence,competitor_problem_solution_better,competitor_comparison_better,has_any_gap,gap_count
0,трусики памперс джуниор,baby care,трусики памперс джуниор,брендовый,"aptekiplus.ru, eapteka.ru, stolichki.ru, ozon....",перекуп,Да,NaN,P&G,Недостаточно авторитетных внешних источников,...,0,0,0,0,1,0,0,0,1,1
1,pampers premium care,baby care,pampers premium care,брендовый,"goldapple.ru, Detmir.ru, votonia.ru, OZON.ru, ...",маркетплейс,Да,NaN,P&G,Недостаточно авторитетных внешних источников,...,0,0,0,0,1,0,0,0,1,1
2,как убрать налет на зубах отзывы,oral care,как убрать налет на зубах отзывы,консультационный,"sprosivracha.com, www.KrasotaiMedicina.ru, alp...",экспертный ресурс,Нет,NaN,конкуренты,"Нет присутствия P&G в AI-ответе, Присутствует ...",...,0,1,0,0,0,0,0,0,1,2
3,гель для душа old spice wolfthorn,hair care,гель для душа old spice wolfthorn,брендовый,"poryadok.ru, lenta.com, goldapple.ru, megaptek...",маркетплейс,Да,NaN,P&G,Недостаточно авторитетных внешних источников,...,0,0,0,0,1,0,0,0,1,1
4,как убрать зеленый налет на зубах,oral care,как убрать зеленый налет на зубах,проблемно-решающий,"probolezny.ru, fdc-vip.ru, доктормамонтов.рф, ...",экспертный ресурс,Нет,NaN,одинаково,"Нет присутствия P&G в AI-ответе, Присутствует ...",...,1,1,0,1,0,0,1,0,1,5


In [70]:
GAP_LABELS = {
    "no_pg_in_ai": "Нет присутствия P&G в AI-ответе",
    "pg_not_leading": "P&G есть, но не в лидирующей рекомендации",
    "no_category_content": "Нет категорийного покрытия",
    "no_consultation_content": "Нет консультационного покрытия",
    "no_comparison_content": "Нет comparison-контента",
    "no_selection_criteria": "Нет объяснения критериев выбора",
    "weak_authority_sources": "Недостаточно авторитетных внешних источников",
    "weak_purchase_platform_presence": "Слабое присутствие на purchase-площадках",
    "competitor_problem_solution_better": "Конкурент сильнее в problem-solution",
    "competitor_comparison_better": "Конкурент сильнее в comparison",
}

gap_summary = (
    df[gap_cols]
    .sum()
    .rename("count")
    .reset_index()
    .rename(columns={"index": "gap"})
)

gap_summary["proportion"] = gap_summary["count"] / len(df)
gap_summary["proportion_pct"] = (gap_summary["proportion"] * 100).round(1)
gap_summary = gap_summary.sort_values("count", ascending=False)
gap_summary["gap_label"] = gap_summary["gap"].map(GAP_LABELS)
gap_summary = gap_summary[["gap", "gap_label", "count", "proportion", "proportion_pct"]]
gap_summary

,gap,gap_label,count,proportion,proportion_pct
0,no_pg_in_ai,Нет присутствия P&G в AI-ответе,221,0.580052,58.0
6,weak_authority_sources,Недостаточно авторитетных внешних источников,205,0.538058,53.8
5,no_selection_criteria,Нет объяснения критериев выбора,131,0.343832,34.4
2,no_category_content,Нет категорийного покрытия,116,0.304462,30.4
3,no_consultation_content,Нет консультационного покрытия,81,0.212598,21.3
8,competitor_problem_solution_better,Конкурент сильнее в problem-solution,75,0.196850,19.7
4,no_comparison_content,Нет comparison-контента,70,0.183727,18.4
1,pg_not_leading,"P&G есть, но не в лидирующей рекомендации",30,0.078740,7.9
7,weak_purchase_platform_presence,Слабое присутствие на purchase-площадках,30,0.078740,7.9
9,competitor_comparison_better,Конкурент сильнее в comparison,10,0.026247,2.6


In [71]:
gap_by_category = (
    df.groupby("категория")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_category

,no_pg_in_ai,pg_not_leading,no_category_content,no_consultation_content,no_comparison_content,no_selection_criteria,weak_authority_sources,weak_purchase_platform_presence,competitor_problem_solution_better,competitor_comparison_better
категория,,,,,,,,,,
baby care,53.9,18.0,30.5,17.2,29.7,34.4,53.9,10.9,18.8,6.2
hair care,63.8,4.7,29.9,22.0,15.0,41.7,56.7,7.1,22.8,1.6
oral care,54.6,0.8,30.3,26.1,8.4,26.9,52.9,5.9,16.8,0.0
other,85.7,0.0,42.9,0.0,42.9,28.6,14.3,0.0,28.6,0.0


In [73]:
gap_by_query_type = (
    df.groupby("тип_запроса")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_query_type

,no_pg_in_ai,pg_not_leading,no_category_content,no_consultation_content,no_comparison_content,no_selection_criteria,weak_authority_sources,weak_purchase_platform_presence,competitor_problem_solution_better,competitor_comparison_better
тип_запроса,,,,,,,,,,
брендовый,9.8,1.5,2.3,0.0,11.3,31.6,88.0,5.3,3.0,0.0
категорийный,78.2,16.5,66.9,0.0,30.1,49.6,54.9,9.0,30.1,4.5
консультационный,91.4,4.3,22.6,79.6,8.6,19.4,14.0,8.6,17.2,2.2
проблемно-решающий,86.4,9.1,13.6,31.8,31.8,22.7,9.1,13.6,68.2,9.1


In [81]:
domains = df[["запрос", "категория", "тип_запроса", "домены_источников"] + gap_cols].copy()

domains["домены_источников"] = (
    domains["домены_источников"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.split(",")
)

domains = domains.explode("домены_источников")
domains["домен"] = domains["домены_источников"].str.strip()
domains = domains[domains["домен"] != ""].copy()

platform_map = {
    "ozon.ru": "маркетплейс",
    "wildberries.ru": "маркетплейс",
    "market.yandex.ru": "маркетплейс",
    "apteka.ru": "аптека/e-com",
    "goldapple.ru": "beauty retail",
    "kp.ru": "медиа/обзоры",
    "vc.ru": "медиа/статьи",
}

domains["platform_type"] = domains["домен"].map(platform_map).fillna("other")

In [82]:
domains

,запрос,категория,тип_запроса,домены_источников,no_pg_in_ai,pg_not_leading,no_category_content,no_consultation_content,no_comparison_content,no_selection_criteria,weak_authority_sources,weak_purchase_platform_presence,competitor_problem_solution_better,competitor_comparison_better,домен,platform_type
0,трусики памперс джуниор,baby care,брендовый,aptekiplus.ru,0,0,0,0,0,0,1,0,0,0,aptekiplus.ru,other
0,трусики памперс джуниор,baby care,брендовый,eapteka.ru,0,0,0,0,0,0,1,0,0,0,eapteka.ru,other
0,трусики памперс джуниор,baby care,брендовый,stolichki.ru,0,0,0,0,0,0,1,0,0,0,stolichki.ru,other
0,трусики памперс джуниор,baby care,брендовый,ozon.ru,0,0,0,0,0,0,1,0,0,0,ozon.ru,маркетплейс
0,трусики памперс джуниор,baby care,брендовый,market.yandex.ru,0,0,0,0,0,0,1,0,0,0,market.yandex.ru,маркетплейс
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,скачать head and shoulders,hair care,брендовый,nsportal.ru,1,0,0,0,0,0,1,0,0,0,nsportal.ru,other
687,скачать head and shoulders,hair care,брендовый,pinkamuz.pro,1,0,0,0,0,0,1,0,0,0,pinkamuz.pro,other
687,скачать head and shoulders,hair care,брендовый,mp3ferz.cc,1,0,0,0,0,0,1,0,0,0,mp3ferz.cc,other
687,скачать head and shoulders,hair care,брендовый,muzmind.org,1,0,0,0,0,0,1,0,0,0,muzmind.org,other


In [83]:
top_domains = domains["домен"].value_counts().head(15).index

gap_by_domain = (
    domains[domains["домен"].isin(top_domains)]
    .groupby("домен")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_domain

,no_pg_in_ai,pg_not_leading,no_category_content,no_consultation_content,no_comparison_content,no_selection_criteria,weak_authority_sources,weak_purchase_platform_presence,competitor_problem_solution_better,competitor_comparison_better
домен,,,,,,,,,,
apteka.ru,45.6,3.5,26.3,8.8,5.3,21.1,66.7,5.3,15.8,0.0
deti.mail.ru,29.2,66.7,29.2,4.2,75.0,25.0,29.2,12.5,29.2,25.0
detmir.ru,23.3,20.0,16.7,3.3,26.7,46.7,90.0,20.0,16.7,10.0
doctorslon.ru,38.5,0.0,38.5,7.7,19.2,34.6,73.1,0.0,19.2,0.0
dzen.ru,76.0,8.0,44.0,40.0,24.0,20.0,20.0,8.0,32.0,4.0
goldapple.ru,45.0,5.0,35.0,2.5,22.5,32.5,70.0,7.5,25.0,0.0
kp.ru,69.2,23.1,28.2,41.0,30.8,28.2,15.4,2.6,33.3,7.7
market.yandex.ru,40.9,6.8,31.8,3.8,13.6,43.9,82.6,7.6,17.4,0.8
ozon.ru,43.0,17.4,32.6,2.3,22.1,50.0,80.2,4.7,14.0,3.5


In [84]:
gap_by_platform_type = (
    domains.groupby("platform_type")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_platform_type

,no_pg_in_ai,pg_not_leading,no_category_content,no_consultation_content,no_comparison_content,no_selection_criteria,weak_authority_sources,weak_purchase_platform_presence,competitor_problem_solution_better,competitor_comparison_better
platform_type,,,,,,,,,,
beauty retail,45.0,5.0,35.0,2.5,22.5,32.5,70.0,7.5,25.0,0.0
other,61.0,9.0,30.2,24.5,19.1,33.2,48.7,9.0,22.5,4.1
аптека/e-com,45.6,3.5,26.3,8.8,5.3,21.1,66.7,5.3,15.8,0.0
маркетплейс,40.3,11.6,32.1,2.6,16.8,49.3,84.3,6.3,15.7,1.5
медиа/обзоры,69.2,23.1,28.2,41.0,30.8,28.2,15.4,2.6,33.3,7.7
медиа/статьи,55.3,34.2,42.1,5.3,63.2,36.8,21.1,7.9,34.2,18.4


In [88]:
### САМОЕ НА МОЙ ВЗГЛЯД ВАЖНОЕ - для "проигрышных запросов". только по кейсам, где выигрывают конкуренты
losing_queries = df[
    df["кто_выигрывает"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("конкуренты")
].copy()

gap_cols = list(GAP_PATTERNS.keys())

losing_gap_summary = (
    losing_queries[gap_cols]
    .mean()
    .mul(100)
    .round(1)
    .rename("gap_rate_pct")
    .reset_index()
    .rename(columns={"index": "gap"})
)

losing_gap_summary["gap_label"] = losing_gap_summary["gap"].map(GAP_LABELS)

losing_gap_summary.sort_values("gap_rate_pct", ascending=False)

,gap,gap_rate_pct,gap_label
0,no_pg_in_ai,82.7,Нет присутствия P&G в AI-ответе
2,no_category_content,55.4,Нет категорийного покрытия
6,weak_authority_sources,47.6,Недостаточно авторитетных внешних источников
5,no_selection_criteria,41.1,Нет объяснения критериев выбора
8,competitor_problem_solution_better,35.1,Конкурент сильнее в problem-solution
4,no_comparison_content,31.0,Нет comparison-контента
1,pg_not_leading,16.1,"P&G есть, но не в лидирующей рекомендации"
3,no_consultation_content,14.3,Нет консультационного покрытия
7,weak_purchase_platform_presence,8.9,Слабое присутствие на purchase-площадках
9,competitor_comparison_better,5.4,Конкурент сильнее в comparison
